# Smart Ear — YAMNet transfer learning

This notebook trains a candidate multi-label classification head on frozen YAMNet embeddings. It does **not** overwrite the Flutter model. Start with ESC-50, then add UrbanSound8K by setting its path below.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import tarfile
import zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/SmartEarTraining')
WORK = Path('/content/smart_ear')
DATASETS = WORK / 'datasets'
PROJECT_ROOT = WORK / 'project'
ARCHIVES = WORK / 'archives'
DATASETS.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVES.mkdir(parents=True, exist_ok=True)

def obtain_archive(name, url):
    destination = ARCHIVES / name
    parts = sorted((DRIVE / 'datasets').glob(f'{name}.part*'))
    direct = DRIVE / 'datasets' / name
    if direct.is_file():
        return direct
    if parts and not destination.is_file():
        print(f'Reassembling {name} from {len(parts)} parts...')
        with destination.open('wb') as output:
            for part in parts:
                with part.open('rb') as source:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
    if not parts and not destination.is_file():
        print(f'Downloading official {name} directly into Colab...')
        subprocess.run(['wget', '-c', '--tries=20', '--timeout=60', url, '-O', str(destination)], check=True)
    return destination

archives = [
    (obtain_archive('UrbanSound8K.tar.gz', 'https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz?download=1'), DATASETS),
    (obtain_archive('ESC-50-master.zip', 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'), DATASETS),
    (DRIVE / 'uploads' / 'audio_training.zip', PROJECT_ROOT),
]
for archive, destination in archives:
    if not archive.is_file():
        raise FileNotFoundError(f'Missing Google Drive upload: {archive}')
    marker = destination / f'.extracted_{archive.stem}'
    if not marker.exists():
        print(f'Extracting {archive.name}...')
        destination.mkdir(parents=True, exist_ok=True)
        if archive.name.endswith('.zip'):
            with zipfile.ZipFile(archive) as source:
                source.extractall(destination)
        else:
            with tarfile.open(archive, 'r:gz') as source:
                source.extractall(destination, filter='data')
        marker.touch()

PROJECT = str(PROJECT_ROOT / 'audio_training')
ESC50 = str(DATASETS / 'ESC-50-master')
URBANSOUND8K = str(DATASETS / 'UrbanSound8K')
CACHE = str(DRIVE / 'cached_embeddings' / 'yamnet_v1.npz')
OUTPUT = str(DRIVE / 'exported_models' / 'yamnet_candidate_v1')
print('Cloud training workspace is ready')

In [ ]:
!pip -q install tensorflow-hub librosa scikit-learn seaborn

import csv, json, os, subprocess
from pathlib import Path
import librosa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics import classification_report, multilabel_confusion_matrix, precision_recall_curve

## Build source-safe manifests
Upload the `audio_training` directory from this repository to `PROJECT`, or clone the repository in Colab and point `PROJECT` at it. Manifests are rebuilt here because paths created on Windows are not valid in Colab.

In [ ]:
manifest_dir = f'{PROJECT}/manifests'
command = ['python', f'{PROJECT}/scripts/build_manifest.py', '--esc50', ESC50, '--output-dir', manifest_dir]
if URBANSOUND8K:
    command += ['--urbansound8k', URBANSOUND8K]
subprocess.run(command, check=True)
subprocess.run(['python', f'{PROJECT}/scripts/audit_manifest.py', f'{manifest_dir}/all.csv', '--fail-on-leakage'], check=True)

In [ ]:
config = json.loads(Path(f'{PROJECT}/config/labels.json').read_text())
labels = config['labels']
label_index = {label: i for i, label in enumerate(labels)}
rows = list(csv.DictReader(open(f'{manifest_dir}/all.csv', encoding='utf-8')))
print(labels)
print(f'{len(rows)} selected recordings')

## Extract and cache YAMNet embeddings
YAMNet expects mono 16 kHz waveform values in `[-1, 1]`. Each clip becomes the mean of its frame-level 1024-value embeddings. Background rows receive an all-zero target vector.

In [ ]:
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
Path(CACHE).parent.mkdir(parents=True, exist_ok=True)

def extract(rows):
    embeddings, targets, splits, paths = [], [], [], []
    for number, row in enumerate(rows, 1):
        waveform, _ = librosa.load(row['path'], sr=16000, mono=True)
        _, frames, _ = yamnet(tf.convert_to_tensor(waveform, tf.float32))
        embeddings.append(tf.reduce_mean(frames, axis=0).numpy())
        target = np.zeros(len(labels), dtype=np.float32)
        if row['label'] != 'background':
            target[label_index[row['label']]] = 1.0
        targets.append(target); splits.append(row['split']); paths.append(row['path'])
        if number % 100 == 0: print(f'Embedded {number}/{len(rows)}')
    return np.asarray(embeddings), np.asarray(targets), np.asarray(splits), np.asarray(paths)

if Path(CACHE).is_file():
    cached = np.load(CACHE, allow_pickle=False)
    X, y, splits, paths = cached['X'], cached['y'], cached['splits'], cached['paths']
else:
    X, y, splits, paths = extract(rows)
    np.savez_compressed(CACHE, X=X, y=y, splits=splits, paths=paths)
print(X.shape, y.shape)

## Train the multi-label classification head

In [ ]:
train = splits == 'train'; validation = splits == 'validation'; test = splits == 'test'
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input((1024,), name='yamnet_embedding'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(labels), activation='sigmoid', name='probabilities'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc')])
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=8, restore_best_weights=True)]
history = model.fit(X[train], y[train], validation_data=(X[validation], y[validation]), epochs=60, batch_size=32, callbacks=callbacks)

## Derive provisional thresholds and evaluate the untouched test split

In [ ]:
validation_scores = model.predict(X[validation], verbose=0)
thresholds = {}
for index, label in enumerate(labels):
    precision, recall, values = precision_recall_curve(y[validation, index], validation_scores[:, index])
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
    best = int(np.nanargmax(f1[:-1])) if len(values) else 0
    thresholds[label] = float(values[best]) if len(values) else 0.5
print(json.dumps(thresholds, indent=2))

test_scores = model.predict(X[test], verbose=0)
test_predictions = np.column_stack([test_scores[:, i] >= thresholds[label] for i, label in enumerate(labels)]).astype(int)
print(classification_report(y[test], test_predictions, target_names=labels, zero_division=0))
matrices = multilabel_confusion_matrix(y[test], test_predictions)
fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3))
for label, matrix, axis in zip(labels, matrices, axes):
    sns.heatmap(matrix, annot=True, fmt='d', ax=axis, cbar=False); axis.set_title(label)
plt.tight_layout()

## Export candidate artifacts
The TFLite file below is the small classification head and expects a `[1, 1024]` YAMNet embedding. It must not replace the current Flutter model by itself.

In [ ]:
output = Path(OUTPUT); output.mkdir(parents=True, exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = converter.convert()
(output / 'smart_ear_yamnet_head.tflite').write_bytes(tflite)
metadata = {
    'version': '2.0.0-candidate', 'model_type': 'yamnet_embedding_head',
    'embedding_size': 1024, 'sample_rate': 16000, 'labels': labels,
    'thresholds': thresholds, 'training_sources': sorted(set(row['source'] for row in rows)),
}
(output / 'model_metadata.json').write_text(json.dumps(metadata, indent=2))
model.save(output / 'smart_ear_yamnet_head.keras')
print(f'Exported candidate artifacts to {output}')